In [1]:
import torch
print(torch.__version__)

print(sys.version)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
2.11.0+cu128
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [21]:
# %pip install pandas
# %pip install numpy
# %pip install -U scikit-learn
# %pip install transformers
# %pip install emoji

In [22]:
import pandas as pd
import numpy as np
import emoji

import json

# SKLearn import
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report # For accuracy score
from sklearn.preprocessing import MultiLabelBinarizer # Convert multiple emotion into format so model can train



In [4]:
# Transformer
import transformers
from transformers import AutoTokenizer
from transformers import XLMRobertaModel # AI Model download

# Import/install training model (XLM-RoBERTa)
model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = XLMRobertaModel.from_pretrained(model_name)

C:\Users\ASUS\.conda\envs\lily-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 498.88it/s]
[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Testing purpose
text = 'I am really happy today!'
inputs = tokenizer(text, return_tensors='pt')
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([1, 8, 768])


In [35]:
df = pd.read_csv('emotion_data.csv')

In [36]:
df.sample(10)

,message,Happy,Sad,Angry,Frustrated,Bored,Excited,Jealous,Insecure,Relieved,Neutral
1477,They said I did a good job on my presentation.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1332,I studied everything and then my brain complet...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1060,I failed something I was supposed to be good at.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
363,Both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1671,I know my kid is not evil or anything. They ar...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
958,That explains why puberty seems so unpredictable.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1568,I do not care. I need my phone.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1455,I hate looking at myself in the mirror.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
804,I am considering replacing my current smartpho...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
724,But they laughed 😭,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Cleaning and data confirming

In [37]:
df.duplicated().sum()

np.int64(101)

In [38]:
df.isna().sum()

message          0
Happy         1737
Sad           1737
Angry         1737
Frustrated    1737
Bored         1737
Excited       1737
Jealous       1737
Insecure      1737
Relieved      1737
Neutral       1737
dtype: int64

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1737 entries, 0 to 1736
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   message     1737 non-null   str    
 1   Happy       0 non-null      float64
 2   Sad         0 non-null      float64
 3   Angry       0 non-null      float64
 4   Frustrated  0 non-null      float64
 5   Bored       0 non-null      float64
 6   Excited     0 non-null      float64
 7   Jealous     0 non-null      float64
 8   Insecure    0 non-null      float64
 9   Relieved    0 non-null      float64
 10  Neutral     0 non-null      float64
dtypes: float64(10), str(1)
memory usage: 149.4 KB


In [40]:
df.shape

(1737, 11)

In [12]:
df.nunique()

message       391
Happy           0
Sad             0
Angry           0
Frustrated      0
Bored           0
Excited         0
Jealous         0
Insecure        0
Relieved        0
Neutral         0
dtype: int64

In [13]:
df.columns

Index(['message', 'Happy', 'Sad', 'Angry', 'Frustrated', 'Bored', 'Excited',
       'Jealous', 'Insecure', 'Relieved', 'Neutral'],
      dtype='str')

In [14]:
df['message'] = df.message.str.strip()

In [27]:
line_break = df['message'].str.contains('\n', regex=False).sum()
print(f"Total message with line break: {line_break}")

Total message with line break: 8


In [29]:
emoji_count_message = df.message.apply(emoji.emoji_count).gt(0).sum()
print(f"Total message containing emoji: {emoji_count_message}")

emoji_count = df.message.apply(emoji.emoji_count).sum()
print(f"Total emoji count: {emoji_count}")

Total message containing emoji: 4
Total emoji count: 6


# Making new dataframe, text only

In [32]:
df_text = df.copy()

In [33]:
df_text['message'] = df_text.message.apply(
    lambda x: emoji.replace_emoji(x, replace='')
)

In [34]:
df_text.message.apply(emoji.emoji_count).sum()

np.int64(0)